In [185]:
import networkx as nx
import plotly.graph_objects as go

In [186]:
G = nx.barabasi_albert_graph(n=200, m=1)

for node in G.nodes:
    nx.set_node_attributes(G, {node: {'text': f"Title: {node}<br>GlobalEventID:{node}<br>Domain: {node}<br>State: {node}<br>" }})  # Replace each {node} here with their respective information from a Pandas dataframe.

# Store ForceAtlas2 positions in object `pos` but also in node attribute 'pos_fa2'
pos = nx.layout.forceatlas2_layout(G, store_pos_as="pos_fa2")

# Repeat, using Spring Layout
pos_s = nx.layout.spring_layout(G, store_pos_as='pos_sp')

# Repeat with random
pos_r = nx.layout.random_layout(G, store_pos_as='pos_r')

In [187]:
def create_graph_traces(G, pos_attribute_name):
    """
    Creates the node trace and edge trace for input graph `G` with nodes having attribute `pos_attribute_name`.

    Args:
        G (nx.Graph) : The graph.
        pos_attribute_name (str) : The attribute name for the node position.
    
    Returns:
        node_trace, edge_trace
    """
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = G.nodes[edge[0]][pos_attribute_name]
        x1, y1 = G.nodes[edge[1]][pos_attribute_name]
        edge_x.append(x0)
        edge_x.append(x1)
        edge_x.append(None)
        edge_y.append(y0)
        edge_y.append(y1)
        edge_y.append(None)

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines')

    node_x = []
    node_y = []
    for node in G.nodes():
        x, y = G.nodes[node][pos_attribute_name]
        node_x.append(x)
        node_y.append(y)

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        marker=dict(
            showscale=True,
            # colorscale options
            #'Greys' | 'YlGnBu' | 'Greens' | 'YlOrRd' | 'Bluered' | 'RdBu' |
            #'Reds' | 'Blues' | 'Picnic' | 'Rainbow' | 'Portland' | 'Jet' |
            #'Hot' | 'Blackbody' | 'Earth' | 'Electric' | 'Viridis' |
            colorscale='YlGnBu_r',
            reversescale=True,
            color=[],
            size=10,
            colorbar=dict(
                thickness=15,
                title=dict(
                text='Node Connections',
                side='right'
                ),
                xanchor='left',
            ),
            line_width=2))

    return node_trace, edge_trace

In [188]:
def color_nodes_by_degree(G, node_trace):
    node_adjacencies = []
    node_text = []
    node_attrs = nx.get_node_attributes(G, name='text')
    for node, adjacencies in enumerate(G.adjacency()):
        node_adjacencies.append(len(adjacencies[1]))
        # node_text.append('# of connections: '+str(len(adjacencies[1])))
        node_text.append(node_attrs[node])

    node_trace.marker.color = node_adjacencies
    node_trace.text = node_text

In [189]:
def create_network_graph(node_trace, edge_trace,
                         title='',
                         annotation_text=''):
    fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title=dict(
                        text=title,
                        font=dict(
                            size=16
                        )
                    ),
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    annotations=[ dict(
                        text=annotation_text,
                        showarrow=False,
                        xref="paper", yref="paper",
                        x=0.005, y=-0.002 ) ],
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                    )
    fig.show()

In [190]:
n_trace_fa2, e_trace_fa2 = create_graph_traces(G, 'pos_fa2')
color_nodes_by_degree(G, n_trace_fa2)
create_network_graph(n_trace_fa2, e_trace_fa2,  title='Barabasi Albert with ForceAtlas2 Layout')


In [191]:
n_trace_sp, e_trace_sp = create_graph_traces(G, 'pos_sp')
color_nodes_by_degree(G, n_trace_sp)
create_network_graph(n_trace_sp, e_trace_sp, title='Barabasi Albert with Spring Layout')

In [192]:
n_trace_r, e_trace_r = create_graph_traces(G, 'pos_r')
color_nodes_by_degree(G, n_trace_r)
create_network_graph(n_trace_r, e_trace_r, title='Barabasi Albert with Random Layout')